#### 工具内部访问 Store（长期记忆）

BaseStore 提供可跨会话持久保存的存储。与 state（短期记忆）不同，保存到该存储中的数据在未来的会话中仍然可用。

In [1]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.chat_models import init_chat_model

# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(('users',), user_id)  # type: ignore
    return str(user_info.value) if user_info else 'Unknown user'

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(('users',), user_id, user_info)  # type: ignore
    return 'Successfully saved user info.'

model = init_chat_model('deepseek-chat')

store = InMemoryStore()
agent = create_agent(model, tools=[get_user_info, save_user_info], store=store)

In [2]:
# First session: save user info
res1 = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev',
            }
        ]
    }
)
print('session 1:')
for msg in res1['messages']:
    msg.pretty_print()

session 1:
================================ Human Message =================================

Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev
================================== Ai Message ==================================

I can help you save that user information. Let me use the save_user_info function to store the details for user abc123.
Tool Calls:
  save_user_info (call_00_I2Jdtj9wSEbPwowcnk2dvyqJ)
 Call ID: call_00_I2Jdtj9wSEbPwowcnk2dvyqJ
  Args:
    user_id: abc123
    user_info: {'name': 'Foo', 'age': 25, 'email': 'foo@langchain.dev'}
================================= Tool Message =================================
Name: save_user_info

Successfully saved user info.
================================== Ai Message ==================================

Great! I've successfully saved the user information for user ID "abc123" with the following details:
- Name: Foo
- Age: 25
- Email: foo@langchain.dev

The user information has been saved successful

In [3]:
# Second session: get user info
res2 = agent.invoke(
    {'messages': [{'role': 'user', 'content': "Get user info for user with id 'abc123'"}]}
)
print('session 2:')
for msg in res2['messages']:
    msg.pretty_print()

session 2:
================================ Human Message =================================

Get user info for user with id 'abc123'
================================== Ai Message ==================================

I'll get the user information for user ID 'abc123' for you.
Tool Calls:
  get_user_info (call_00_P2elcztQksF6ehlgMlKIWH1U)
 Call ID: call_00_P2elcztQksF6ehlgMlKIWH1U
  Args:
    user_id: abc123
================================= Tool Message =================================
Name: get_user_info

{'name': 'Foo', 'age': 25, 'email': 'foo@langchain.dev'}
================================== Ai Message ==================================

Here's the user information for user ID 'abc123':

- **Name**: Foo
- **Age**: 25
- **Email**: foo@langchain.dev
